# Bayesian Learning Pipeline for Client Personas

notebook combines Bayesian Gaussian Mixture modeling with Learning Vector Quantization (LVQ) to identify and interpret distinct customer personas from banking client data.

## Objectives

- Prepare and encode mixed numerical/categorical client data.
- Discover latent persona clusters using Bayesian Gaussian Mixture.
- Build business-readable cluster profiles.
- Train and interpret an LVQ model to obtain representative class prototypes.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report
from sklearn.mixture import BayesianGaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklvq.models import GLVQ

plt.rcParams["figure.dpi"] = 150
plt.style.use("ggplot")

## 1) Data Loading and Preprocessing

We keep the preprocessing logic explicit because clustering quality depends heavily on feature treatment.

- Categorical variables are one-hot encoded.
- Continuous variables are standardized with `StandardScaler` for Bayesian Gaussian Mixture.


In [ ]:
mapping_dicts = {
    "Gender": {"0": "Male", "1": "Female", "0.0": "Male", "1.0": "Female"},
    "Job": {
        "1": "Unemployed",
        "2": "Employee/Worker",
        "3": "Manager/Executive",
        "4": "Entrepreneur/Freelancer",
        "5": "Retired",
    },
    "Area": {"1": "Nord", "2": "Centro", "3": "Sud/Isole"},
    "CitySize": {"1": "Small town", "2": "Medium-sized city", "3": "Large city"},
    "Investments": {
        "1": "No investments",
        "2": "Mostly lump sum",
        "3": "Capital accumulation",
    },
}

# Load data and remove the ID key because it has no clustering signal.
data = pd.read_excel("./Dataset1_BankClients.xlsx")
data = data.drop(columns=["ID"])

categorical_columns = ["Gender", "Job", "Area", "CitySize", "Investments"]

In [ ]:
# One-hot encode categorical features.
data_encoded = pd.get_dummies(data, columns=categorical_columns, dtype=int)

# Continuous numeric columns from original raw data (excluding coded categoricals).
continuous_columns = [
    col
    for col in data.select_dtypes(include=["number"]).columns
    if col not in categorical_columns
]

# Standard scaling for Bayesian Gaussian Mixture.
scaler_1 = StandardScaler()
data_scaled_bayesian = data_encoded.copy()
data_scaled_bayesian[continuous_columns] = scaler_1.fit_transform(
    data_scaled_bayesian[continuous_columns]
)

data_scaled_bayesian.head()

## 2) Bayesian Gaussian Mixture for Persona Discovery

We fit a Dirichlet Process Bayesian GMM with an upper bound on components, then keep only components with meaningful weight (greater than 5%). This acts as automatic model complexity control.


In [ ]:
max_clusters = 15

bgm = BayesianGaussianMixture(
    n_components=max_clusters,
    covariance_type="full",
    weight_concentration_prior_type="dirichlet_process",
    random_state=42,
    max_iter=1000,
    init_params="kmeans",
)

bgm.fit(data_scaled_bayesian)

raw_labels = bgm.predict(data_scaled_bayesian)
raw_weights = pd.Series(bgm.weights_, name="weight")

weight_threshold = 0.05
active_components = raw_weights[raw_weights > weight_threshold].index.tolist()
active_mask = pd.Series(raw_labels).isin(active_components)

data_persona = data_scaled_bayesian.loc[active_mask].copy()
data_persona["persona_cluster"] = pd.Series(raw_labels)[active_mask].values

# Remap sparse component IDs to consecutive persona labels.
cluster_remap = {old: new for new, old in enumerate(sorted(active_components))}
data_persona["persona_cluster"] = data_persona["persona_cluster"].map(cluster_remap)

persona_summary = (
    data_persona["persona_cluster"]
    .value_counts(normalize=True)
    .sort_index()
    .rename("share")
    .to_frame()
)
persona_summary["count"] = (
    data_persona["persona_cluster"].value_counts().sort_index().values
)

print(f"Max components (K): {max_clusters}")
print(f"Retained persona frameworks: {len(active_components)}")
print(f"Active original component IDs: {sorted(active_components)}")
persona_summary

In [ ]:
# Persist the clustered dataset for downstream LVQ modeling.
data_persona.to_excel("result/bayesian-clustered-bankClients.xlsx", index=False)
data_persona.head()

## 3) Qualitative Persona Overlay

To make clusters actionable for business stakeholders, we convert centroids back to interpretable attributes:

- Continuous means are inverse-transformed to original units.
- Dominant categorical values are recovered from one-hot proportions.
- Confidence tags indicate whether each dominant category is clearly represented.


In [ ]:
cluster_scaled_means = data_persona.groupby("persona_cluster").mean(numeric_only=True)

continuous_scaled = cluster_scaled_means[continuous_columns]
continuous_original = pd.DataFrame(
    scaler_1.inverse_transform(continuous_scaled),
    columns=continuous_columns,
    index=continuous_scaled.index,
)

categorical_profile = pd.DataFrame(index=cluster_scaled_means.index)
for col in categorical_columns:
    onehot_cols = [c for c in cluster_scaled_means.columns if c.startswith(f"{col}_")]
    if not onehot_cols:
        continue

    proportions = cluster_scaled_means[onehot_cols]
    dominant_onehot = proportions.idxmax(axis=1)
    dominant_value_raw = dominant_onehot.str.replace(f"{col}_", "", regex=False)
    dominant_value_mapped = dominant_value_raw.map(
        lambda x: mapping_dicts.get(col, {}).get(str(x), x)
    )

    categorical_profile[f"{col}_dominant"] = dominant_value_mapped
    categorical_profile[f"{col}_share"] = proportions.max(axis=1)

business_profile = continuous_original.join(categorical_profile)
cluster_count = data_persona["persona_cluster"].value_counts().sort_index()
cluster_share = cluster_count / cluster_count.sum()
business_profile["persona_size"] = (
    cluster_count.reindex(business_profile.index).fillna(0).astype(int)
)
business_profile["persona_share"] = cluster_share.reindex(
    business_profile.index
).fillna(0.0)


def share_confidence_tag(v: float) -> str:
    if v >= 0.75:
        return "High confidence"
    if v >= 0.55:
        return "Medium confidence"
    return "Low confidence"


for col in categorical_columns:
    share_col = f"{col}_share"
    if share_col in business_profile.columns:
        business_profile[f"{col}_confidence"] = business_profile[share_col].apply(
            share_confidence_tag
        )

business_profile

## Final Clustering Result (5 Personas)

The final clustering solution identifies **5 customer personas** (total \(N = 4456\)).

### Persona 0: Mature Mainstream Working Families

- **Size:** 2288 clients (**51.35%**)
- **Profile:** Average age around 59, medium family size, medium income/wealth, moderate debt.
- **Behavior:** Balanced financial behavior, moderate digital usage, moderate financial education.
- **Dominant traits:** Female, Employee/Worker, Nord, Medium-sized city.
- **Investment style:** Capital accumulation.

### Persona 1: Urban Middle-Aged Female Accumulators

- **Size:** 704 clients (**15.80%**)
- **Profile:** Around 53 years old, smaller families, slightly higher income/wealth than Persona 0.
- **Behavior:** Higher digital adoption and more active lifestyle.
- **Dominant traits:** Female, mostly Employee/Worker, strongly Nord, Large city.
- **Investment style:** Capital accumulation.

### Persona 2: Elderly Male Retirees (Conservative)

- **Size:** 357 clients (**8.01%**)
- **Profile:** Oldest segment (about 81), small households, low income/wealth, very low debt.
- **Behavior:** Low digital engagement, low lifestyle/luxury orientation.
- **Dominant traits:** Male, Retired, Nord, Medium-sized city.
- **Investment style:** Mostly lump sum.

### Persona 3: Affluent Digital Professionals

- **Size:** 678 clients (**15.22%**)
- **Profile:** Around 52 years old, highest income/wealth among all clusters.
- **Behavior:** Highest financial education, strongest digital adoption, high ESG and lifestyle/luxury orientation.
- **Dominant traits:** Mostly Male, Employee/Worker, strongly Nord, Large city.
- **Investment style:** Capital accumulation.

### Persona 4: Elderly Female Retirees (Low-Intensity Banking)

- **Size:** 429 clients (**9.63%**)
- **Profile:** About 82 years old, low income/wealth, low debt.
- **Behavior:** Low digital activity and low lifestyle/luxury propensity.
- **Dominant traits:** Female, Retired, Nord, Medium-sized city.
- **Investment style:** Mostly lump sum.

### Overall Interpretation

The clustering is mainly driven by:

1. **Life stage** (working-age vs retired)
2. **Economic capacity** (mass market vs affluent)
3. **Behavioral engagement** (digital/active vs low-intensity)
4. **Retired segment split by gender** (Persona 2 vs Persona 4)

This segmentation provides a strong basis for targeted strategies in advisory, digital engagement, and product personalization.


## 4) LVQ Modeling on Persona Labels

The LVQ stage provides prototype vectors per persona. These prototypes are useful to summarize each segment and to visualize class geometry.


In [ ]:
df = pd.read_excel("./result/bayesian-clustered-bankClients.xlsx")
X = df.drop(columns=["persona_cluster"])
y = df["persona_cluster"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

lvq = GLVQ(
    prototype_init="class-conditional-mean",
    prototype_n_per_class=1,
    random_state=42,
    solver_type="steepest-gradient-descent",
    solver_params={"max_runs": 100},
)
lvq.fit(X_train, y_train)

y_pred_test = lvq.predict(X_test)
y_pred_train = lvq.predict(X_train)

print("Model Evaluation")
print(f"Training accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Testing accuracy: {accuracy_score(y_test, y_pred_test):.4f}")
print("\nClassification Report")
print(classification_report(y_test, y_pred_test))

In [ ]:
prototypes = lvq.prototypes_
prototype_labels = getattr(lvq, "prototypes_labels_", np.sort(np.unique(y_train)))

if hasattr(X_train, "columns"):
    feature_names = list(X_train.columns)
else:
    feature_names = [f"feature_{i}" for i in range(prototypes.shape[1])]

if len(feature_names) != prototypes.shape[1]:
    feature_names = [f"feature_{i}" for i in range(prototypes.shape[1])]

prototype_df = pd.DataFrame(prototypes, columns=feature_names)
prototype_df.insert(0, "Prototype_Label", prototype_labels)
prototype_df.head()

## 5) Decision Tree Model for Inference

To enable fast operational scoring, we train a Decision Tree on the persona labels and provide an inference helper that accepts a feature dictionary.

- The model is trained on the same feature space used by LVQ.
- Inputs are automatically aligned to training columns.
- The function returns predicted persona and class probabilities.


In [ ]:
dt_model = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
)
dt_model.fit(X_train, y_train)

dt_pred_test = dt_model.predict(X_test)
dt_pred_train = dt_model.predict(X_train)

print("Decision Tree Evaluation")
print(f"Training accuracy: {accuracy_score(y_train, dt_pred_train):.4f}")
print(f"Testing accuracy: {accuracy_score(y_test, dt_pred_test):.4f}")
print("\nClassification Report")
print(classification_report(y_test, dt_pred_test))

feature_order = list(X_train.columns)


def infer_persona_decision_tree(input_features: dict) -> dict:
    row = pd.DataFrame([input_features]).reindex(columns=feature_order, fill_value=0)
    pred = int(dt_model.predict(row)[0])
    proba = dt_model.predict_proba(row)[0]
    class_probs = {int(cls): float(p) for cls, p in zip(dt_model.classes_, proba)}
    return {
        "predicted_persona": pred,
        "confidence": float(np.max(proba)),
        "class_probabilities": class_probs,
    }


# Example inference using one record from the test set.
sample_payload = X_test.iloc[0].to_dict()
infer_persona_decision_tree(sample_payload)

In [ ]:
# Decision Tree diagram (limited depth for readability).
plt.figure(figsize=(24, 12))
plot_tree(
    dt_model,
    feature_names=feature_order,
    class_names=[str(c) for c in dt_model.classes_],
    filled=True,
    rounded=True,
    impurity=True,
    proportion=True,
    max_depth=3,
    fontsize=8,
)
plt.title("Decision Tree Diagram for Persona Inference (Depth <= 3)")
plt.tight_layout()
plt.savefig("result/persona_decision_tree_diagram.png", dpi=300, bbox_inches="tight")
plt.show()

## 6) Prototype Interpretation and Visualization

Two complementary views are provided:

- A feature heatmap of the prototype signatures.
- A 2D PCA projection to compare customer points vs. learned prototypes.


In [ ]:
plt.figure(figsize=(14, 10))
heatmap_data = prototype_df.set_index("Prototype_Label").T
sns.heatmap(
    heatmap_data,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f",
    cbar_kws={"label": "Feature Value / Centroid Weight"},
)

plt.title("LVQ Prototypes: Feature Signatures for each Persona", fontsize=16, pad=20)
plt.xlabel("Persona (Prototype Label)", fontsize=12)
plt.ylabel("Features", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_train_pca = pca.fit_transform(X_train)
prototypes_pca = pca.transform(prototypes)

plt.figure(figsize=(12, 8))
sns.scatterplot(
    x=X_train_pca[:, 0],
    y=X_train_pca[:, 1],
    hue=y_train,
    palette="tab10",
    alpha=0.3,
    legend="full",
    edgecolor=None,
)

plt.scatter(
    prototypes_pca[:, 0],
    prototypes_pca[:, 1],
    c="black",
    marker="X",
    s=250,
    linewidths=2,
    edgecolors="white",
    label="LVQ Prototypes",
    zorder=5,
)

plt.title("LVQ Prototypes and Client Data in 2D PCA Space", fontsize=16)
plt.xlabel(f"PCA Component 1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PCA Component 2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.legend(title="Persona Class", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
prototype_df.to_csv("result/lvq_prototypes_interpreted.csv", index=False)
prototype_df

## 7) Conclusion

This merged notebook now provides a single reproducible pipeline:

1. Data preprocessing for mixed-type features.
2. Bayesian persona discovery with automatic component pruning.
3. Business-readable persona overlay.
4. LVQ prototype learning and visualization for interpretability.
5. Decision Tree training for fast persona inference.

You can run top-to-bottom to regenerate all intermediate and final artifacts.
